# Session 03 — Data Preprocessing & Feature Engineering
**UKUDLA Machine Learning Certificate**

In this notebook, we will load the Smart Farming 2024 (SF24) dataset, clean it, and engineer agronomically meaningful features based on the 4-step framework.

In [ ]:
# 1. Import needed Libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Set visual style
sns.set_theme(style="whitegrid", palette="viridis")
print("Libraries loaded successfully.")

Libraries loaded successfully.


### Step 1: Load and Inspect the Data

In [ ]:
# Upload the dataset (from the local directory where you saved it)
df = pd.read_csv('sf24data.csv')
print(f"Initial Shape: {df.shape}")

Initial Shape: (2310, 23)


In [ ]:
# CRITICAL: Drop duplicates. This Kaggle dataset has thousands of them.
df = df.drop_duplicates()
print(f"Shape after dropping duplicates: {df.shape}")

df.head()

Shape after dropping duplicates: (2200, 23)


,N,P,K,temperature,humidity,ph,rainfall,label,soil_moisture,soil_type,...,organic_matter,irrigation_frequency,crop_density,pest_pressure,fertilizer_usage,growth_stage,urban_area_proximity,water_source_type,frost_risk,water_usage_efficiency
0,90.0,42.0,43.0,20.879744,82.002744,6.502985,202.935536,rice,29.446064,2.0,...,3.121395,4.0,NaN,57.607308,188.194958,1.0,2.719614,3.0,95.649985,1.193293
1,85.0,58.0,41.0,21.770462,80.319644,7.038096,226.655537,rice,12.851183,3.0,...,2.142021,4.0,NaN,NaN,70.963629,1.0,4.714427,2.0,77.265694,1.752672
2,60.0,55.0,44.0,NaN,82.320763,7.840207,263.964248,rice,29.363913,2.0,...,1.474974,1.0,12.654395,1.034478,191.976077,1.0,30.431736,2.0,18.192168,3.035541
3,74.0,35.0,40.0,26.491096,80.158363,6.980401,242.864034,rice,26.207732,3.0,...,8.393907,1.0,10.864360,24.091888,55.761388,3.0,10.861071,3.0,82.818720,1.273341
4,78.0,42.0,42.0,20.130175,81.604873,7.628473,262.717340,rice,28.236236,2.0,...,NaN,3.0,13.852910,38.811481,185.259702,2.0,47.190777,3.0,25.466499,2.578671


### Step 2: Data Cleaning - Missing Values
In real life, sensors fail. We will simulate missing values in the `rainfall` column to demonstrate how to handle them using the **Median**.

In [ ]:
# Simulate missing values for demonstration
df.loc[0:10, 'rainfall'] = np.nan
print("Missing values before:", df['rainfall'].isnull().sum())

# Why Median? Because means are skewed by extreme weather.
median_rainfall = df['rainfall'].median()
df['rainfall'] = df['rainfall'].fillna(median_rainfall)

print(f"Imputed missing values with median: {median_rainfall:.2f}")
print("Missing values after:", df['rainfall'].isnull().sum())

Missing values before: 121
Imputed missing values with median: 94.33
Missing values after: 0


### Step 3: Data Cleaning - Outliers (IQR Method)
We will cap extreme values in the `rainfall` column so they don't break the model.

In [ ]:
# Calculate IQR
Q1 = df['rainfall'].quantile(0.25)
Q3 = df['rainfall'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR

print(f"Q1: {Q1:.2f} | Q3: {Q3:.2f} | IQR: {IQR:.2f}")
print(f"Upper Bound for Rainfall: {upper_bound:.2f}")

# Cap the outliers
df['rainfall'] = np.where(df['rainfall'] > upper_bound, upper_bound, df['rainfall'])
print("Outliers above the upper bound have been capped.")

Q1: 65.52 | Q3: 119.65 | IQR: 54.13
Upper Bound for Rainfall: 200.86
Outliers above the upper bound have been capped.


### Step 4: Feature Engineering - The 4 Methods
**Method 1: Interaction Feature (THI)**
Captures combined heat stress. High Temp + High Humidity is dangerous.

In [ ]:
# Formula: THI = (1.8T + 32) - (0.55 - 0.0055H)(1.8T - 26)
df['THI'] = (1.8 * df['temperature'] + 32) - \
            (0.55 - 0.0055 * df['humidity']) * \
            (1.8 * df['temperature'] - 26)

print(df[['temperature', 'humidity', 'THI']].head())

   temperature   humidity        THI
0    20.879744  82.002744  68.436943
1    21.770462  80.319644  69.759463
2          NaN  82.320763        NaN
3    26.491096  80.158363  77.317623
4    20.130175  81.604873  67.198876


In [ ]:
print("Dataset shape:", df.shape)
print("Number of rows:", len(df))
print("Number of crop labels:", df['label'].nunique())
print("Missing THI values:", df['THI'].isna().sum())

In [ ]:
# Visualizing THI against Crop Type
plt.figure(figsize=(12, 5))
sns.boxplot(data=df, x='label', y='THI')
plt.xticks(rotation=90)
plt.title('Temperature-Humidity Index (THI) Across Different Crops')
plt.show()

**Method 2: Ratio Feature (N:P Ratio)**
Captures nutrient balance. A plant needs both N and P to thrive.

In [ ]:
# Formula: N / (P + 1) to prevent division by zero
df['N_P_Ratio'] = df['N'] / (df['P'] + 1)
print(df[['N', 'P', 'N_P_Ratio']].head())

**Method 3: Aggregation Feature (Total NPK)**
A quick snapshot of overall soil fertility.

In [ ]:
df['Total_NPK'] = df['N'] + df['P'] + df['K']
print(df[['N', 'P', 'K', 'Total_NPK']].head())

**Method 4: Domain Index (Growing Degree Days)**
Assuming a base temperature of 10°C for crop growth.

In [ ]:
# GDD = max(Temp - Base_Temp, 0)
df['GDD'] = np.maximum(df['temperature'] - 10, 0)
print(df[['temperature', 'GDD']].head())

### Step 5: Categorical Encoding
Convert `soil_type` (1, 2, 3) into binary columns.

In [ ]:
# One-hot encoding with drop_first=True to avoid the dummy variable trap
df = pd.get_dummies(df, columns=['soil_type'], drop_first=True, prefix='Soil')
print("Columns after encoding:")
print(df.columns.tolist())

### Step 6: The Danger of Too Many Features (Multicollinearity)
We check for any features with a correlation > 0.85 and drop them.

In [ ]:
# Calculate correlation matrix (only numerical columns)
numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr().abs()

# Select upper triangle of correlation matrix
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Find features to drop
to_drop = [column for column in upper.columns if any(upper[column] > 0.85)]

if len(to_drop) > 0:
    print(f"Features to drop due to high collinearity: {to_drop}")
    df_cleaned = df.drop(columns=to_drop)
    print(f"Shape before: {df.shape} | Shape after: {df_cleaned.shape}")
else:
    print("No features exceeded the 0.85 correlation threshold.")
    df_cleaned = df

### Final Check
The dataset is now clean, enriched with agronomic features, and ready for modeling in Session 4.

In [ ]:
df_cleaned.head()